# Householder Reflection for Solving $A \mathbf{x} = b$

## Problem Statement
Solve the $3 \times 3$ linear system for unknowns $\mathbf{x} = [x, y, z]^T$:
$$A \mathbf{x} = b$$

where:
$$A = \begin{bmatrix} 3 & 4 & 5 \\ -1 & 1 & -3 \\ 2 & -2 & 3 \end{bmatrix}, \quad b = \begin{bmatrix} 7 \\ 1 \\ 5 \end{bmatrix}$$

---

## Geometric Intuition & Lecture Notation: $u = x - x'$

Instead of arbitrary formulas, Householder reflection has a direct geometric meaning:
1. **The Target Vector $x'$:** We want to transform a column (or subcolumn) $x$ into a target vector $x'$ that preserves length ($\|x'\| = \|x\|$) but has zeros in all lower entries:
   $$x = \begin{bmatrix} x_0 \\ x_1 \\ \vdots \end{bmatrix} \quad \xrightarrow{\text{reflect}} \quad x' = \begin{bmatrix} \pm \|x\| \\ 0 \\ \vdots \end{bmatrix} = \begin{bmatrix} \pm \alpha \\ 0 \\ \vdots \end{bmatrix}$$
2. **The Reflection Normal $u = x - x'$:** The mirror plane bisects the angle between $x$ and $x'$. The normal vector perpendicular to the mirror is simply the difference:
   $$u = x - x'$$
3. **The Reflection Matrix $H$:**
   $$H = I - 2 \frac{u u^T}{u^T u}$$
4. **Why $H x = x'$ (Proof):**
   Since $\|x'\| = \|x\|$, expanding $u^T u = (x - x')^T(x - x')$ gives:
   $$u^T u = \|x\|^2 - 2 x^T x' + \|x'\|^2 = 2 \|x\|^2 - 2 x^T x' = 2 x^T (x - x') = 2 u^T x \implies \frac{u^T x}{u^T u} = \frac{1}{2}$$
   Applying $H$ directly gives:
   $$H x = \left(I - 2 \frac{u u^T}{u^T u}\right) x = x - 2 u \left(\frac{u^T x}{u^T u}\right) = x - 2 u \left(\frac{1}{2}\right) = x - u = x - (x - x') = x'$$
   The reflection maps $x$ **directly and exactly onto $x'$**.

> **Sign Choice for Numerical Stability:** To avoid catastrophic subtraction of similar numbers when forming $u = x - x'$, we choose the top entry of $x'$ with sign opposite to $x_0$:
> $$x'_0 = -\text{sign}(x_0) \|x\| \implies u_0 = x_0 - (-\text{sign}(x_0)\|x\|) = x_0 + \text{sign}(x_0)\|x\|$$


In [1]:
import numpy as np

# Define 3x3 system A x = b
A = np.array([
    [ 3.0,  4.0,  5.0],
    [-1.0,  1.0, -3.0],
    [ 2.0, -2.0,  3.0]
])

b = np.array([7.0, 1.0, 5.0])

A_orig = A.copy()
b_orig = b.copy()

print('Original Matrix A:')
print(A)
print('\nOriginal Vector b:')
print(b)


Original Matrix A:
[[ 3.  4.  5.]
 [-1.  1. -3.]
 [ 2. -2.  3.]]

Original Vector b:
[7. 1. 5.]


### Step 1: Zero Out Column 0

1. **Input vector:** $x = A[:, 0] = \begin{bmatrix} 3 \\ -1 \\ 2 \end{bmatrix}$
2. **Target vector $x'$:** Zero out rows 1 and 2 while preserving length:
   $$\alpha_1 = \|x\| = \sqrt{3^2 + (-1)^2 + 2^2} = \sqrt{14} \approx 3.7417$$
   $$x' = \begin{bmatrix} -\alpha_1 \\ 0 \\ 0 \end{bmatrix} \approx \begin{bmatrix} -3.7417 \\ 0 \\ 0 \end{bmatrix}$$
3. **Householder normal vector $u_1 = x - x'$:**
   $$u_1 = \begin{bmatrix} 3 \\ -1 \\ 2 \end{bmatrix} - \begin{bmatrix} -\sqrt{14} \\ 0 \\ 0 \end{bmatrix} = \begin{bmatrix} 3 + \sqrt{14} \\ -1 \\ 2 \end{bmatrix} \approx \begin{bmatrix} 6.7417 \\ -1.0000 \\ 2.0000 \end{bmatrix}$$
4. **Form $H_1 = I_3 - 2 \frac{u_1 u_1^T}{u_1^T u_1}$ and apply:** $A^{(1)} = H_1 A$, $b^{(1)} = H_1 b$.


In [2]:
# Step 1: Eliminate rows 1 and 2 in column 0
x = A[:, 0]
alpha1 = np.linalg.norm(x)

# Target vector x' with length preserved and zeros below first element
sign1 = -1.0 if x[0] >= 0 else 1.0
x_prime = np.zeros_like(x)
x_prime[0] = sign1 * alpha1

# Normal vector u1 = x - x'
u1 = x - x_prime

# Householder Matrix H1 = I - 2 * (u u^T) / (u^T u)
H1 = np.eye(3) - 2.0 * np.outer(u1, u1) / np.dot(u1, u1)

# Update A and b
A1 = H1 @ A
b1 = H1 @ b

print(f'Input vector x: {x}')
print(f'Target vector x\': {np.round(x_prime, 4)}')
print(f'Householder vector u1 = x - x\': {np.round(u1, 4)}')
print('\nMatrix A^(1) after Step 1 (Rows 1 & 2 of Col 0 zeroed out):')
print(np.round(A1, 4))
print('\nVector b^(1) after Step 1:')
print(np.round(b1, 4))


Input vector x: [ 3. -1.  2.]
Target vector x': [-3.7417  0.      0.    ]
Householder vector u1 = x - x': [ 6.7417 -1.      2.    ]

Matrix A^(1) after Step 1 (Rows 1 & 2 of Col 0 zeroed out):
[[-3.7417 -1.8708 -6.4143]
 [ 0.      1.8708 -1.3069]
 [-0.     -3.7417 -0.3862]]

Vector b^(1) after Step 1:
[-8.0178  3.2276  0.5448]


### Step 2: Zero Out Column 1

To preserve row 0 and column 0, we focus on the $2 \times 2$ submatrix below row 1:
1. **Subvector $x$:** $x = A^{(1)}[1:3, 1] = \begin{bmatrix} A^{(1)}_{11} \\ A^{(1)}_{21} \end{bmatrix} \approx \begin{bmatrix} 1.8708 \\ -3.7417 \end{bmatrix}$
2. **Target subvector $x'$:** Zero out row 2 while preserving length:
   $$\alpha_2 = \|x\| = \sqrt{(1.8708)^2 + (-3.7417)^2} \approx 4.1833$$
   $$x' = \begin{bmatrix} -\alpha_2 \\ 0 \end{bmatrix} \approx \begin{bmatrix} -4.1833 \\ 0 \end{bmatrix}$$
3. **Householder normal vector $u_2 = x - x'$:**
   $$u_2 = x - x' = \begin{bmatrix} 1.8708 \\ -3.7417 \end{bmatrix} - \begin{bmatrix} -4.1833 \\ 0 \end{bmatrix} = \begin{bmatrix} 1.8708 + 4.1833 \\ -3.7417 \end{bmatrix} = \begin{bmatrix} 6.0541 \\ -3.7417 \end{bmatrix}$$
4. **Sub-reflection $H_{2,\text{sub}} = I_2 - 2 \frac{u_2 u_2^T}{u_2^T u_2}$, embedded into full $3 \times 3$:**
   $$H_2 = \begin{bmatrix} 1 & 0 & 0 \\ 0 & & \\ 0 & & H_{2,\text{sub}} \end{bmatrix}$$
5. **Upper-Triangular Result:** $R = H_2 A^{(1)} = H_2 H_1 A$ and $\tilde{b} = H_2 b^{(1)} = H_2 H_1 b$.


In [3]:
# Step 2: Eliminate element at row 2, col 1
# x is the 2D subvector of column 1 from row 1 down
x_sub = A1[1:, 1]
alpha2 = np.linalg.norm(x_sub)

# Target subvector x_prime with row 2 zeroed out
sign2 = -1.0 if x_sub[0] >= 0 else 1.0
x_sub_prime = np.zeros_like(x_sub)
x_sub_prime[0] = sign2 * alpha2

# Normal vector u2 = x - x'
u2 = x_sub - x_sub_prime

# 2x2 Sub-reflection matrix
H2_sub = np.eye(2) - 2.0 * np.outer(u2, u2) / np.dot(u2, u2)

# Embed into full 3x3 matrix (row 0 and col 0 unchanged)
H2 = np.eye(3)
H2[1:, 1:] = H2_sub

# Final upper-triangular matrix R and transformed vector b_tilde
R = H2 @ A1
b_tilde = H2 @ b1

print(f'Subvector x: {np.round(x_sub, 4)}')
print(f'Target subvector x\': {np.round(x_sub_prime, 4)}')
print(f'Householder vector u2 = x - x\': {np.round(u2, 4)}')
print('\nUpper-Triangular Matrix R = H2 @ H1 @ A:')
print(np.round(R, 4))
print('\nTransformed Vector b_tilde = H2 @ H1 @ b:')
print(np.round(b_tilde, 4))


Subvector x: [ 1.8708 -3.7417]
Target subvector x': [-4.1833  0.    ]
Householder vector u2 = x - x': [ 6.0541 -3.7417]

Upper-Triangular Matrix R = H2 @ H1 @ A:
[[-3.7417 -1.8708 -6.4143]
 [-0.     -4.1833  0.239 ]
 [-0.      0.     -1.3416]]

Transformed Vector b_tilde = H2 @ H1 @ b:
[-8.0178 -0.9562  3.1305]


## Back-Substitution Algorithm

The transformed system $R \mathbf{x} = \tilde{b}$ is upper-triangular:
$$\begin{bmatrix} R_{00} & R_{01} & R_{02} \\ 0 & R_{11} & R_{12} \\ 0 & 0 & R_{22} \end{bmatrix} \begin{bmatrix} x \\ y \\ z \end{bmatrix} = \begin{bmatrix} \tilde{b}_0 \\ \tilde{b}_1 \\ \tilde{b}_2 \end{bmatrix}$$

Solve from bottom to top:
1. **Solve for $z$ (row 2):**
   $$z = \frac{\tilde{b}_2}{R_{22}} = \frac{3.1305}{-1.3416} = -\frac{7}{3} \approx -2.33333333$$
2. **Solve for $y$ (row 1):**
   $$y = \frac{\tilde{b}_1 - R_{12} z}{R_{11}} = \frac{-0.9562 - (0.2390)(-\frac{7}{3})}{-4.1833} = \frac{2}{21} \approx 0.09523810$$
3. **Solve for $x$ (row 0):**
   $$x = \frac{\tilde{b}_0 - R_{01} y - R_{02} z}{R_{00}} = \frac{128}{21} \approx 6.09523810$$


In [4]:
def back_substitution(R, b):
    """Solves upper-triangular system R x = b."""
    n = len(b)
    x = np.zeros(n, dtype=np.float64)
    for i in range(n - 1, -1, -1):
        if np.isclose(R[i, i], 0.0):
            raise ValueError(f"Singular matrix: zero pivot at row {i}")
        known_sum = np.dot(R[i, i + 1:], x[i + 1:])
        x[i] = (b[i] - known_sum) / R[i, i]
    return x

sol = back_substitution(R, b_tilde)
print('Computed Solution [x, y, z]:')
print(sol)

print('\nExact Fractions:')
print(f'x = 128/21 = {128/21:.8f}')
print(f'y =   2/21 = {2/21:.8f}')
print(f'z =  -7/3  = {-7/3:.8f}')

# Verification against original system
residual = A_orig @ sol - b_orig
print('\nVerification:')
print('A @ sol =', A_orig @ sol)
print('b_orig  =', b_orig)
print('Residual norm ||A x - b|| =', np.linalg.norm(residual))


Computed Solution [x, y, z]:
[ 6.0952381   0.0952381  -2.33333333]

Exact Fractions:
x = 128/21 = 6.09523810
y =   2/21 = 0.09523810
z =  -7/3  = -2.33333333

Verification:
A @ sol = [7. 1. 5.]
b_orig  = [7. 1. 5.]
Residual norm ||A x - b|| = 1.9860273225978185e-15


---

## Comparison: Householder Reflections vs. Givens Rotations

| Feature | Givens Rotations | Householder Reflections |
| :--- | :--- | :--- |
| **Elimination Unit** | Zeroes **one element** at a time ($A_{ij} \rightarrow 0$) | Zeroes **an entire sub-column** at a time ($u = x - x'$) |
| **Transformations for $3 \times 3$** | 3 rotations ($G_{10}, G_{20}, G_{21}$) | **2 reflections** ($H_1, H_2$) |
| **Transformations for $n \times n$** | $\frac{n(n-1)}{2} = \mathcal{O}(n^2)$ rotations | **$n-1 = \mathcal{O}(n)$ reflections** |
| **Flop Count** | $\approx \frac{4}{3} n^3$ flops | $\approx \frac{2}{3} n^3$ flops (half the work for dense matrices!) |
| **Best Use Case** | Sparse matrices, banded systems, updating QR | Dense matrices, full QR decomposition |
